In [ ]:
#structured output 

"""models can be requested to prvide their response in a format matching a given schema. This is useful for ensuring the output can be easily
parsed and used in subsequent processing"""

#Pydantic

# pydantic models provide the richest feature set with field validation, description and nested structues

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001E71C230A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E71C1F2810>, model_name='qwen/qwen-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")


In [13]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)
model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E71C34D250>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E71C635C40>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'param

In [11]:
model.invoke("Provide me the details of the movie Harry Potter")

AIMessage(content="**Harry Potter Movie Series Overview**\n\nThe Harry Potter movie series is a collection of eight fantasy films based on the seven novels by J.K. Rowling. The series follows the journey of its titular character, Harry Potter, a young wizard who attends Hogwarts School of Witchcraft and Wizardry.\n\n**Main Plot**\n\nThe story begins with the introduction of Harry Potter (played by Daniel Radcliffe), an orphan boy who lives with his cruel and neglectful Muggle (non-magical) relatives, the Dursleys. On his eleventh birthday, Harry receives a letter from Hogwarts, and his life is forever changed. He discovers that he is a wizard and begins attending Hogwarts, where he makes friends with Ron Weasley (Rupert Grint) and Hermione Granger (Emma Watson).\n\nThroughout the series, Harry and his friends become entangled in a world of magic, mystery, and danger. They must confront the dark wizard, Lord Voldemort (Ralph Fiennes), who murdered Harry's parents and seeks to dominate t

In [14]:

model_with_structure.invoke("Provide me the detail of the movie Harry Potter")
 

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ypk3txw7t', 'function': {'arguments': '{"director":"Chris Columbus","rating":7.6,"title":"Harry Potter","year":2001}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 280, 'total_tokens': 312, 'completion_time': 0.07666537, 'completion_tokens_details': None, 'prompt_time': 0.015600806, 'prompt_tokens_details': None, 'queue_time': 0.287592358, 'total_time': 0.092266176}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ecc27-0795-7542-880d-318cab94ac3f-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Chris Columbus', 'rating': 7.6, 'title': 'Harry Potter', 'year': 2001}, 'id': 'ypk3txw7t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 280, 'output_tokens': 32, 

In [15]:
#nested
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    budget: float | None = Field(None, description="Budget is million USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details of the movie Harry Potter")

print(response)

title='Harry Potter' year=2001 cast=[Actor(name='Daniel Radcliffe', role='Harry Potter'), Actor(name='Rupert Grint', role='Ron Weasley'), Actor(name='Emma Watson', role='Hermione Granger')] budget=125.0


In [16]:
#TYPE DICT

"""TypeDict provides a simpler alternative using python's built-in typing, ideal when you dont need runtime validation"""

from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str, ..., "The director of the movie"]
    year: Annotated[int, ...,"The year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie"]


model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide me the details of the movie Harry Potter")
response

{'director': 'Chris Columbus',
 'rating': 7.6,
 'title': 'Harry Potter',
 'year': 2001}

In [17]:
#model with type dict

model.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

In [ ]:
###data classes


"""A data class is a class typically contaning mainly dara, although there are not really any restrictions, you create it suing the @dataclasses decorator"""



In [5]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from dataclasses import dataclass


@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str


llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

agent = create_agent(
    model=llm,                 # pass model instance
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from John Doe, patel@gmail.com, +219980234567"
        }
    ]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='patel@gmail.com', phone='+219980234567')


In [10]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from dataclasses import dataclass


class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

agent = create_agent(
    model=llm,                 # pass model instance
    response_format=Movie
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract the movie details of Hulk"
        }
    ]
})

print(result["structured_response"])

title='Hulk' year=2003 director='Ang Lee' rating=5.6
